In [10]:
import pandas as pd
import numpy as np
import json

In [ ]:
with open("../../../data/test/4_complete_featured_data.jsonl", "r") as f:
    conversations = [json.loads(c) for c in f.readlines()]

conversations_df = pd.DataFrame(conversations)

features_df = pd.json_normalize(conversations_df["features"])
embeddings_df = pd.DataFrame(
    conversations_df["embeddings"].tolist(),
    columns=[f"emb_{i}" for i in range(len(conversations_df["embeddings"].iloc[0]))]
)

X_train = pd.concat([features_df, embeddings_df], axis=1)

Y_train = conversations_df["true_label"]

### Verify the shape of both dataframes is the same

In [12]:
X_train.shape[0] == Y_train.shape[0]

True

### Mark categorical columns

In [13]:
categorical_cols = ["guest_persona_hint", "price_sensitivity", "language"]
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")

### Encode Labels

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(Y_train)

### Split training set from validation set

In [15]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

### Train model

In [16]:
import lightgbm as lgb

train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=categorical_cols)

params = {
    "objective": "multiclass",
    "num_class": 4,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "min_data_in_leaf": 10,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(50)],
)

Training until validation scores don't improve for 30 rounds
[50]	training's multi_logloss: 0.0780409	valid_1's multi_logloss: 0.37989
[100]	training's multi_logloss: 0.00675034	valid_1's multi_logloss: 0.264568
[150]	training's multi_logloss: 0.000573492	valid_1's multi_logloss: 0.230699
Early stopping, best iteration is:
[154]	training's multi_logloss: 0.000476936	valid_1's multi_logloss: 0.226314


### Evaluate model

In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_pred_proba = model.predict(X_val, num_iteration=model.best_iteration)
y_pred = np.argmax(y_pred_proba, axis=1)

print(classification_report(y_val, y_pred, target_names=le.classes_))
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

   abandoned       0.89      0.96      0.92        25
        high       0.88      0.92      0.90        25
         low       1.00      1.00      1.00        25
      medium       0.95      0.84      0.89        25

    accuracy                           0.93       100
   macro avg       0.93      0.93      0.93       100
weighted avg       0.93      0.93      0.93       100

[[24  0  0  1]
 [ 2 23  0  0]
 [ 0  0 25  0]
 [ 1  3  0 21]]


### Cross-validate with multiple runs of the model

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in skf.split(X_train, y_encoded):
    train_data = lgb.Dataset(X_train.iloc[train_idx], label=y_encoded[train_idx], categorical_feature=categorical_cols)
    val_data = lgb.Dataset(X_train.iloc[val_idx], label=y_encoded[val_idx], reference=train_data, categorical_feature=categorical_cols)

    m = lgb.train(params, train_data, num_boost_round=500,
                   valid_sets=[val_data], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

    preds = np.argmax(m.predict(X_train.iloc[val_idx], num_iteration=m.best_iteration), axis=1)
    scores.append(f1_score(y_encoded[val_idx], preds, average="macro"))

print(f"Mean macro-F1: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[147]	valid_0's multi_logloss: 0.241306
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[280]	valid_0's multi_logloss: 0.243532
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[109]	valid_0's multi_logloss: 0.346698
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[136]	valid_0's multi_logloss: 0.326672
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[153]	valid_0's multi_logloss: 0.331947
Mean macro-F1: 0.884 ± 0.025


### Check feature importance

In [19]:
importance = pd.DataFrame({
    "feature": model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print(importance.head(20))

                      feature   importance
0     mentions_specific_dates  1491.891430
8        reached_payment_step  1226.760781
512                   emb_496   506.140443
1          mentions_room_type   285.229018
54                     emb_38   253.257357
6       attractions_mentioned   247.310409
450                   emb_434   224.729993
100                    emb_84   141.353078
1431                 emb_1415   133.629083
553                   emb_537   124.691803
10          price_sensitivity   123.827249
1430                 emb_1414   119.393801
1257                 emb_1241   116.815721
287                   emb_271   108.935395
1315                 emb_1299    94.619949
162                   emb_146    77.629671
597                   emb_581    72.394015
644                   emb_628    67.059953
355                   emb_339    66.079051
114                    emb_98    66.073925


In [ ]:
import pickle

artifact = {
    "model": model,
    "label_encoder": le,
    "feature_columns": list(X_train.columns),
    "categorical_columns": categorical_cols,
}

with open("../../../data/test/booking_intent_artifact.pkl", "wb") as f:
    pickle.dump(artifact, f)